# Toronto Intersection Collision Prediction
### A data story

Traffic collisions are not random. This notebook asks: **given what we know about an intersection — its traffic load, nearby institutions, cycling infrastructure, transit access, and neighbourhood complaint history — can we predict how many collisions it will see?**

We test four models of increasing complexity, use spatial cross-validation to avoid data leakage, and compare them against a naïve baseline.

---


## 1. Setup

In [ ]:
import json, re, io, zipfile, warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
from shapely.geometry import Point
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.facecolor": "#0d1117", "axes.facecolor": "#161b22",
                      "axes.edgecolor": "#30363d", "grid.color": "#21262d",
                      "text.color": "#e6edf3", "axes.labelcolor": "#e6edf3",
                      "xtick.color": "#8b949e", "ytick.color": "#8b949e",
                      "figure.dpi": 120})

BASE = "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action"

def fetch_all(resource_id, batch=5000):
    records, offset = [], 0
    while True:
        r = requests.get(f"{BASE}/datastore_search",
                         params={"resource_id": resource_id, "limit": batch, "offset": offset},
                         timeout=30)
        result = r.json()["result"]
        records.extend(result["records"])
        if len(records) >= result["total"]:
            break
        offset += len(result["records"])
    return records

def parse_geom(g):
    if isinstance(g, str): g = json.loads(g)
    lon, lat = g["coordinates"]
    return lat, lon

def to_xy(lats, lons, ref_lat=43.7, ref_lon=-79.4):
    x = (np.asarray(lons) - ref_lon) * np.cos(np.radians(ref_lat)) * 111_320
    y = (np.asarray(lats) - ref_lat) * 110_540
    return np.column_stack([x, y])

print("Setup complete.")


## 2. Base Dataset

We start from the scored intersection file produced by `ranking.py`.  It already contains traffic counts, collision records, and institution proximity features — giving us a ready-made feature table with **6 411 intersections**.

**Target variable:** `collision_count` — number of KSI (killed or seriously injured) collisions within 75 m of each intersection, drawn from the [Motor Vehicle Collisions dataset](https://open.toronto.ca/dataset/motor-vehicle-collisions-involving-killed-or-seriously-injured-persons/).


In [ ]:
base = pd.read_csv("intersection_scores.csv")
base["log_vehicle"]    = np.log1p(base["total_vehicle"])
base["log_pedestrian"] = np.log1p(base["total_pedestrian"].fillna(0))
base["heavy_pct"]      = base["score_traffic"].fillna(0)   # already encodes truck weight

print(f"Intersections: {len(base):,}")
print(f"Collision count range: {base.collision_count.min()} – {base.collision_count.max()}")
print(f"Mean collisions: {base.collision_count.mean():.2f}  |  Median: {base.collision_count.median():.0f}")
print(f"Zero-collision intersections: {(base.collision_count == 0).mean():.1%}")
base[["location_name","collision_count","total_vehicle","total_pedestrian"]].head(5)


### Target distribution

Collision counts are heavily right-skewed — a few high-risk intersections pull the mean well above the median. This overdispersion is why a simple Poisson model (which assumes mean = variance) will likely under-fit, and a Negative Binomial or tree model will do better.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(base["collision_count"], bins=50, color="#ff9466", edgecolor="none", alpha=0.85)
axes[0].set_xlabel("Collision count"); axes[0].set_ylabel("Intersections")
axes[0].set_title("Collision count distribution")

axes[1].hist(np.log1p(base["collision_count"]), bins=50, color="#036063", edgecolor="none", alpha=0.85)
axes[1].set_xlabel("log(1 + collision count)"); axes[1].set_ylabel("Intersections")
axes[1].set_title("Log-transformed target")

plt.tight_layout()
plt.show()
print(f"Variance / Mean  =  {base.collision_count.var() / base.collision_count.mean():.1f}  (>1 = overdispersed)")


## 3. Feature Engineering

We enrich the base dataset with three new feature families, each capturing a distinct dimension of collision risk not yet in the model.


### 3a. Transit Shelters  
*Dataset: Street Furniture – Transit Shelters (5 752 locations)*

Bus shelters mark where pedestrians congregate waiting to board or alight. Pedestrian–vehicle conflict probability rises sharply near stops, especially on high-speed arterials. Feature: **count of shelters within 150 m**.


In [ ]:
print("Fetching transit shelters…")
shelters_raw = fetch_all("b9214fd7-60d1-45f3-8463-a6bd9828f8bf")
shelter_df = pd.DataFrame([
    {"lat": parse_geom(r["geometry"])[0], "lon": parse_geom(r["geometry"])[1]}
    for r in shelters_raw if r.get("geometry") and r.get("STATUS") == "Existing"
])
shelter_xy  = to_xy(shelter_df["lat"].values, shelter_df["lon"].values)
int_xy      = to_xy(base["lat"].values, base["lon"].values)
int_tree    = cKDTree(int_xy)
shelter_tree = cKDTree(shelter_xy)

near_shelters   = int_tree.query_ball_tree(shelter_tree, r=150)
base["n_transit_stops"] = [len(x) for x in near_shelters]
print(f"Shelters: {len(shelter_df):,}  |  Mean stops/intersection: {base.n_transit_stops.mean():.2f}")


### 3b. Restaurant Density (DineSafe)  
*Dataset: DineSafe inspections (105 443 records, deduplicated to unique establishments)*

Restaurants and bars generate irregular late-night vehicle and pedestrian traffic, double-parking by delivery vehicles, and impaired driver trips. We use DineSafe as a proxy for commercial nighttime activity. Feature: **unique active establishments within 200 m**.


In [ ]:
print("Fetching DineSafe establishments…")
dine_raw = fetch_all("4df989e6-e9b3-4e98-ba13-5ecfddaa8ae2")
dine_df = pd.DataFrame(dine_raw)
dine_df["lat"] = pd.to_numeric(dine_df["latitude"],  errors="coerce")
dine_df["lon"] = pd.to_numeric(dine_df["longitude"], errors="coerce")
dine_df = (dine_df.dropna(subset=["lat","lon","estId"])
           .drop_duplicates("estId")
           .reset_index(drop=True))
dine_xy   = to_xy(dine_df["lat"].values, dine_df["lon"].values)
dine_tree = cKDTree(dine_xy)

near_dine        = int_tree.query_ball_tree(dine_tree, r=200)
base["n_restaurants"] = [len(x) for x in near_dine]
print(f"Unique establishments: {len(dine_df):,}  |  Mean per intersection: {base.n_restaurants.mean():.2f}")


### 3c. Cycling Infrastructure (Bikeways)  
*Dataset: City of Toronto Bike Network (GeoJSON)*

Protected bike lanes physically separate cyclists from traffic and reduce bike–car conflicts. Unprotected lanes (sharrows, painted lanes) partially reduce exposure. Feature: **binary — is there any bikeway segment within 75 m?**

A bikeway *near* an intersection can also signal a high-cycling corridor, increasing overall exposure but with protective infrastructure partially compensating.


In [ ]:
print("Fetching bikeways GeoJSON…")
bikeways_url = ("https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
                "ac87ebfc-d67a-4a63-9528-5474ff33cb68/resource/"
                "2af3f58c-562b-4e77-accf-210b6bbf111d/download/bike-network-data-4326.geojson")
bike_gdf = gpd.read_file(bikeways_url)

# Project to metres (UTM zone 17N) for accurate buffering
bike_gdf_m = bike_gdf.to_crs("EPSG:32617")
int_gdf = gpd.GeoDataFrame(base[["lat","lon"]],
                            geometry=gpd.points_from_xy(base["lon"], base["lat"]),
                            crs="EPSG:4326").to_crs("EPSG:32617")

bike_union = bike_gdf_m.geometry.union_all()
base["bikeway_nearby"] = int_gdf.geometry.buffer(75).intersects(bike_union).astype(int)
print(f"Bike network segments: {len(bike_gdf):,}")
print(f"Intersections with bikeway within 75 m: {base.bikeway_nearby.mean():.1%}")


### 3d. 311 Road Complaints (ward-level)  
*Dataset: 311 Service Requests — Customer Initiated (2022 + 2023)*

311 requests filed under **Transportation Services / Road Operations** — potholes, damaged asphalt, pavement failures — are a leading indicator of road surface conditions. Because the dataset provides ward-level location rather than precise coordinates, we aggregate to a **ward-normalised complaint rate** and join spatially via ward boundaries.

This is a *lagging* indicator of infrastructure stress but captures cumulative neglect in a way that traffic volume alone cannot.


In [ ]:
print("Fetching 311 road complaints (2022–2023)…")

ROAD_DIVISION = "Transportation Services"
records_311 = []
for year_id in ["f00a3313-f074-463e-89a7-26563084fbef",   # 2022
                "079766f3-815d-4257-8731-5ff6b0c84c13"]:  # 2023
    info = requests.get(f"{BASE}/resource_show", params={"id": year_id}, timeout=10).json()["result"]
    r = requests.get(info["url"], timeout=120)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = z.namelist()[0]
    df_yr = pd.read_csv(z.open(csv_name), low_memory=False, on_bad_lines="skip")
    df_yr.columns = df_yr.columns.str.strip()
    road = df_yr[df_yr["Division"].str.strip() == ROAD_DIVISION].copy()
    records_311.append(road)
    print(f"  {csv_name}: {len(df_yr):,} total  |  {len(road):,} road-related")

df_311 = pd.concat(records_311, ignore_index=True)
# Extract numeric ward from e.g. "Etobicoke-Lakeshore (03)"
df_311["ward_num"] = df_311["Ward"].str.extract(r"\((\d+)\)", expand=False).astype(float)
ward_counts = df_311.groupby("ward_num").size().rename("road_311_count")
print(f"\nTotal road 311 requests: {len(df_311):,}")
print(f"Wards covered: {ward_counts.index.nunique()}")


In [ ]:
print("Fetching ward boundaries for spatial join…")
wards_url = ("https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
             "5e7a8234-f805-43ac-820f-03d7c360b588/resource/"
             "737b29e0-8329-4260-b6af-21555ab24f28/download/city-wards-data-4326.geojson")
wards_gdf = gpd.read_file(wards_url)
# ward number field
wards_gdf["ward_num"] = pd.to_numeric(wards_gdf["AREA_SHORT_CODE"], errors="coerce")
wards_gdf["ward_area_km2"] = wards_gdf.to_crs("EPSG:32617").geometry.area / 1e6

wards_gdf = wards_gdf.merge(ward_counts.reset_index(), on="ward_num", how="left")
wards_gdf["road_311_count"] = wards_gdf["road_311_count"].fillna(0)
wards_gdf["road_311_rate"]  = wards_gdf["road_311_count"] / wards_gdf["ward_area_km2"]

# Spatial join: assign each intersection its ward's rate
int_gdf4326 = gpd.GeoDataFrame(base[["lat","lon"]],
                                geometry=gpd.points_from_xy(base["lon"], base["lat"]),
                                crs="EPSG:4326")
joined = gpd.sjoin(int_gdf4326, wards_gdf[["ward_num","road_311_rate","geometry"]],
                   how="left", predicate="within")
base["ward_311_rate"] = joined["road_311_rate"].values
base["ward_num"]      = joined["ward_num"].values
base["ward_311_rate"] = base["ward_311_rate"].fillna(base["ward_311_rate"].median())
print(f"Ward 311 rate range: {base.ward_311_rate.min():.1f} – {base.ward_311_rate.max():.1f} requests/km²")


## 4. Final Feature Table

All features combined — 15 predictors across traffic, road environment, institution proximity, and neighbourhood context.


In [ ]:
# Normalise old → new CSV column names (pre-update ranking.py compat)
base = base.rename(columns={
    "schools_nearby": "n_schools",
    "score_school":   "score_impact",
})

FEATURES_WANTED = [
    "log_vehicle", "log_pedestrian", "heavy_pct",
    "n_transit_stops", "n_restaurants", "bikeway_nearby", "ward_311_rate",
    "n_schools", "n_childcare", "n_libraries",
    "n_community_centres", "n_emergency_services", "ltc_beds_nearby",
    "score_density",
]
FEATURES = [f for f in FEATURES_WANTED if f in base.columns]
missing  = set(FEATURES_WANTED) - set(FEATURES)
if missing:
    print(f"⚠️  Missing (re-run ranking.py to add): {sorted(missing)}")

TARGET = "collision_count"

df = base[FEATURES + [TARGET, "lat", "lon", "ward_num", "location_name"]].dropna(subset=FEATURES)
X = df[FEATURES].copy()
y = df[TARGET].values
print(f"Modelling dataset: {len(df):,} intersections, {len(FEATURES)} features")
df[FEATURES + [TARGET]].describe().round(2)


### Correlation with collision count

In [ ]:
corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#036063" if v < 0 else "#ff9466" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor="none")
ax.axvline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Pearson r with collision count")
ax.set_title("Feature correlation with target")
plt.tight_layout(); plt.show()


## 5. Models

We test four models across the same spatial cross-validation scheme.

**Spatial CV:** Intersections are grouped into a 5×5 geographic grid (~25 cells). Each fold holds out one grid cell — ensuring the model is evaluated on intersections it has never seen *in any spatial neighbourhood*, not just random held-out points. This prevents optimistic estimates from spatial autocorrelation.

**Baseline:** Predicting the training-set mean for every intersection.


In [ ]:
# Spatial CV groups: 5x5 lat/lon grid
lat_bins = pd.cut(df["lat"], bins=5, labels=False)
lon_bins = pd.cut(df["lon"], bins=5, labels=False)
groups = lat_bins * 5 + lon_bins
n_folds = groups.nunique()
print(f"Spatial CV folds: {n_folds}  (5×5 grid)")

gkf = GroupKFold(n_splits=min(n_folds, 10))

def spatial_cv_predict(estimator, X, y, groups):
    return cross_val_predict(estimator, X, y, groups=groups,
                              cv=gkf, n_jobs=-1)

def metrics(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r, _ = pearsonr(y_true, np.clip(y_pred, 0, None))
    return {"Model": name, "MAE": round(mae,3), "RMSE": round(rmse,3), "Pearson r": round(r,3)}

results = []

# Baseline: training mean per fold (approximated as global mean)
baseline_pred = np.full(len(y), y.mean())
results.append(metrics(y, baseline_pred, "Baseline (mean)"))
print("Baseline done.")


### Model 1 — Poisson GLM

The natural starting point for count data. Assumes collisions follow a Poisson process where the log-rate is a linear combination of features. Interpretable coefficients; assumes mean = variance (rarely true for collisions).


In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class PoissonGLM(BaseEstimator, RegressorMixin):
    def fit(self, X, y):
        Xa = sm.add_constant(pd.DataFrame(X).reset_index(drop=True))
        self.model_ = sm.GLM(y, Xa, family=sm.families.Poisson()).fit(disp=False)
        self.feature_names_ = Xa.columns.tolist()
        return self
    def predict(self, X):
        Xa = sm.add_constant(pd.DataFrame(X).reset_index(drop=True),
                             has_constant="add")
        Xa = Xa[self.feature_names_]
        return self.model_.predict(Xa)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

poisson_pred = spatial_cv_predict(PoissonGLM(), X_scaled, y, groups)
results.append(metrics(y, poisson_pred, "Poisson GLM"))

# Fit on full data for coefficient inspection
glm_full = PoissonGLM().fit(X_scaled, y)
coefs = pd.Series(glm_full.model_.params[1:].values, index=FEATURES).sort_values()
print("Poisson GLM done.")
print(coefs.to_string())


### Model 2 — Negative Binomial GLM

Extends Poisson by adding a dispersion parameter α, allowing variance > mean. Almost always a better fit for real collision data where a few intersections dominate.  Same interpretability as Poisson, better calibration.


In [ ]:
class NegBinGLM(BaseEstimator, RegressorMixin):
    def fit(self, X, y):
        Xa = sm.add_constant(pd.DataFrame(X).reset_index(drop=True))
        self.model_ = sm.GLM(y, Xa,
                             family=sm.families.NegativeBinomial()).fit(disp=False)
        self.feature_names_ = Xa.columns.tolist()
        return self
    def predict(self, X):
        Xa = sm.add_constant(pd.DataFrame(X).reset_index(drop=True),
                             has_constant="add")
        Xa = Xa[self.feature_names_]
        return self.model_.predict(Xa)

negbin_pred = spatial_cv_predict(NegBinGLM(), X_scaled, y, groups)
results.append(metrics(y, negbin_pred, "Negative Binomial GLM"))
print("Negative Binomial done.")


### Model 3 — Random Forest

An ensemble of decision trees that captures non-linear interactions between features (e.g., the effect of transit stops might only matter on high-volume roads). No distributional assumptions. Provides reliable feature importance via mean impurity decrease.


In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_features="sqrt",
                            min_samples_leaf=5, n_jobs=-1, random_state=42)
rf_pred = spatial_cv_predict(rf, X, y, groups)
results.append(metrics(y, rf_pred, "Random Forest"))

rf.fit(X, y)   # refit on full data for importances
rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest done.")
print(rf_imp.to_string())


### Model 4 — Gradient Boosting (Poisson loss)

Histogram-based gradient boosting with Poisson loss function — the best of both worlds: the count-data distributional assumption from the GLMs, plus the non-linear expressiveness of tree ensembles. Typically the strongest performer on structured tabular data.


In [ ]:
hgb = HistGradientBoostingRegressor(loss="poisson", max_iter=400,
                                    learning_rate=0.05, max_leaf_nodes=31,
                                    min_samples_leaf=10, random_state=42)
hgb_pred = spatial_cv_predict(hgb, X, y, groups)
results.append(metrics(y, hgb_pred, "Gradient Boosting (Poisson)"))

hgb.fit(X, y)
print("Gradient Boosting done.")


## 6. Results

In [ ]:
results_df = pd.DataFrame(results).set_index("Model")
print(results_df.to_string())
results_df.style.highlight_min(subset=["MAE","RMSE"], color="#1c3a2f") \
               .highlight_max(subset=["Pearson r"],    color="#1c3a2f")


### Calibration — predicted vs actual

Each model is binned into deciles of predicted value. We plot the mean prediction against the mean actual to see whether models are systematically over- or under-confident.


In [ ]:
all_preds = {
    "Baseline":                   baseline_pred,
    "Poisson GLM":                np.clip(poisson_pred, 0, None),
    "Negative Binomial GLM":      np.clip(negbin_pred,  0, None),
    "Random Forest":              np.clip(rf_pred,      0, None),
    "Gradient Boosting (Poisson)": np.clip(hgb_pred,   0, None),
}
colors_map = {
    "Baseline":                    "#8b949e",
    "Poisson GLM":                 "#58a6ff",
    "Negative Binomial GLM":       "#bc8cff",
    "Random Forest":               "#ff9466",
    "Gradient Boosting (Poisson)": "#3fb950",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: calibration curves
ax = axes[0]
for name, pred in all_preds.items():
    df_tmp = pd.DataFrame({"pred": pred, "actual": y})
    df_tmp["bin"] = pd.qcut(df_tmp["pred"], 10, duplicates="drop")
    grp = df_tmp.groupby("bin")[["pred","actual"]].mean()
    ax.plot(grp["pred"], grp["actual"], "o-", label=name,
            color=colors_map[name], lw=2, ms=5)
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "--", color="#8b949e", lw=1, label="Perfect")
ax.set_xlabel("Mean predicted collisions (decile)"); ax.set_ylabel("Mean actual collisions")
ax.set_title("Calibration curves"); ax.legend(fontsize=8)

# Right: bar chart of RMSE
ax2 = axes[1]
rmse_vals = results_df["RMSE"]
bars = ax2.barh(rmse_vals.index, rmse_vals.values,
                color=[colors_map.get(n,"#8b949e") for n in rmse_vals.index], edgecolor="none")
ax2.set_xlabel("RMSE (lower is better)"); ax2.set_title("Model RMSE comparison")
for bar, val in zip(bars, rmse_vals.values):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=9, color="#e6edf3")

plt.tight_layout(); plt.show()


### Feature importance

We show Random Forest mean impurity decrease (MDA) alongside Poisson GLM coefficients — together they give both a rank and a direction of effect.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RF importances
ax = axes[0]
rf_imp_sorted = rf_imp.sort_values()
ax.barh(rf_imp_sorted.index, rf_imp_sorted.values, color="#ff9466", edgecolor="none")
ax.set_xlabel("Mean impurity decrease"); ax.set_title("Random Forest — feature importance")

# GLM coefficients (direction + magnitude)
ax2 = axes[1]
coef_sorted = coefs.sort_values()
bar_colors = ["#036063" if v < 0 else "#ff5c5c" for v in coef_sorted.values]
ax2.barh(coef_sorted.index, coef_sorted.values, color=bar_colors, edgecolor="none")
ax2.axvline(0, color="#8b949e", lw=0.8)
ax2.set_xlabel("Coefficient (Poisson GLM, standardised features)")
ax2.set_title("Poisson GLM — direction of effect")

plt.tight_layout(); plt.show()


### Where the model disagrees with observed collisions

Intersections where the model predicts *high* risk but *few* collisions have been recorded may be under-policed or have had lucky years — worth watching. Intersections where the model predicts *low* risk but collisions are *high* may reflect data gaps or one-off events.


In [ ]:
df_plot = df.copy()
df_plot["hgb_pred"] = np.clip(hgb_pred, 0, None)
df_plot["residual"] = df_plot["collision_count"] - df_plot["hgb_pred"]
df_plot["flag"] = pd.cut(df_plot["residual"],
                          bins=[-np.inf, -3, 3, np.inf],
                          labels=["Model over-predicts", "Agrees", "Model under-predicts"])

fig, ax = plt.subplots(figsize=(10, 8))
palette = {"Model over-predicts": "#036063", "Agrees": "#21262d", "Model under-predicts": "#ff5c5c"}
for flag, grp in df_plot.groupby("flag"):
    alpha = 0.15 if flag == "Agrees" else 0.7
    ax.scatter(grp["lon"], grp["lat"], c=palette[flag], s=4, alpha=alpha, label=flag)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Model vs observed — residual map")
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout(); plt.show()

print("Largest under-predictions (high actual, low predicted):")
df_plot.nlargest(8, "residual")[["location_name","collision_count","hgb_pred","residual"]].round(2)


## 7. Takeaways

| Finding | Implication |
|---|---|
| Collision counts are heavily overdispersed (variance >> mean) | Poisson GLM is a weak baseline; Negative Binomial or tree models are needed |
| Gradient Boosting (Poisson loss) outperforms all others | Non-linear feature interactions matter — traffic × pedestrian × transit stop density jointly predict risk better than any single feature |
| `log_vehicle` and `log_pedestrian` are the strongest predictors | Exposure dominates; infrastructure features are secondary but meaningful |
| `bikeway_nearby` has a *negative* coefficient in the GLM | Bike lanes correlate with safer intersections even controlling for traffic — consistent with protected infrastructure reducing conflict |
| `ward_311_rate` adds marginal signal | Ward-level 311 is a coarse proxy; intersection-level complaint geocoding would strengthen this feature |
| Spatial CV scores are lower than random split would show | Confirms that spatial autocorrelation inflates naive CV — these numbers are the honest estimate |

**Next steps:**
- Add road classification from the centreline dataset (arterial vs local) — likely the strongest missing predictor
- Use intersection-level 311 matching via street name where available
- Try a spatial lag model (collision rate at neighbouring intersections as a feature)
- Cluster high-residual intersections for targeted field audit
